In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Load dataset with severity and resolution scores
df = pd.read_csv(
    "../data/geocoded_data/data_severity_resolution_score_assigned.csv"
)

# keep valid coordinates
df = df[df["ok"] == True]
df = df.dropna(subset=["lat", "lng"])

print("Total valid points:", df.shape)

C:\Users\shrey\AppData\Local\Temp\ipykernel_67940\2198664400.py:6: DtypeWarning: Columns (3,4,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


Total valid points: (413131, 28)


In [2]:
geometry = [Point(xy) for xy in zip(df["lng"], df["lat"])]

gdf_points = gpd.GeoDataFrame(
    df,
    geometry=geometry,
    crs="EPSG:4326"   # lat/lon
)

gdf_points.head()

,Opened,Description,Address,Zip Code,Closed Date 1,Closed Date 2,Status,Number,closed,resolution_time_hours,...,Description_norm,domain,issue_type,domain_weight,issue_weight,severity_score,expected_resolution,resolution_factor,resolution_score,geometry
0,2021-02-03 13:30:00,Information on How to Request an Account Adjus...,"4501 Dudley LN NW, Atlanta, 30327",30327,2021-09-13 11:28:00,NaN,Resolved,CS0063095,2021-09-13 11:28:00,4677.982222,...,information on how to request an account adjus...,Administrative / Account Services,Service Request,2,2,4,551.416860,8.483568,1,POINT (-84.3882 33.87756)
1,2021-03-03 13:30:00,Information on How to Dispute Your Water and S...,"238 peachtree cir, ATLANTA, 30309",30309,2021-07-15 12:54:00,NaN,Resolved,CS0060088,2021-07-15 12:54:00,3215.405833,...,information on how to dispute your water and s...,Water & Sewer,Billing / Account,5,1,5,398.083287,8.077219,1,POINT (-84.38632 33.79538)
2,2021-03-03 13:30:00,Information on How to Dispute Your Water and S...,"3434 Habersham Rd NW, Atlanta, 30305",30305,2022-05-26 09:54:00,NaN,Resolved,CS0026984,2022-05-26 09:54:00,10772.409440,...,information on how to dispute your water and s...,Water & Sewer,Billing / Account,5,1,5,398.083287,27.060693,1,POINT (-84.39063 33.84864)
3,2021-09-03 13:30:00,Street Light Bulb Replacement or Street Light ...,"1160 Veltrie Circle, ATLANTA, 30311",30311,2021-04-14 15:24:00,NaN,Resolved,CS0001758,2021-04-14 15:24:00,865.910278,...,street light bulb replacement or street light ...,Road & Infrastructure,Service Failure,4,3,12,3329.238548,0.260093,5,POINT (-84.47501 33.7234)
4,2021-11-03 13:30:00,Right of Way Maintenance Visibility/Overgrowth...,"2855 elliott cir , ATLANTA, 30305",30305,NaN,2025-03-06 17:47:00,Resolved,CS0056768,2025-06-03 17:47:00,37084.283330,...,right of way maintenance visibility/overgrowth...,Road & Infrastructure,Information / Other,4,1,4,1997.484941,18.565488,1,POINT (-84.36737 33.83342)


In [3]:
import os
import requests
import zipfile

url = "https://www2.census.gov/geo/tiger/TIGER2020/TABBLOCK20/tl_2020_13_tabblock20.zip"

zip_path = "tl_2020_13_tabblock20.zip"
out_dir = "tl_2020_13_tabblock20"

# download
if not os.path.exists(zip_path):
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            if chunk:
                f.write(chunk)

# unzip
if not os.path.exists(out_dir):
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)

# read blocks
blocks_ga = gpd.read_file(
    os.path.join(out_dir, "tl_2020_13_tabblock20.shp")
)

print(blocks_ga.columns)

Index(['STATEFP20', 'COUNTYFP20', 'TRACTCE20', 'BLOCKCE20', 'GEOID20',
       'GEOIDFQ20', 'NAME20', 'MTFCC20', 'UR20', 'UACE20', 'FUNCSTAT20',
       'ALAND20', 'AWATER20', 'INTPTLAT20', 'INTPTLON20', 'HOUSING20', 'POP20',
       'geometry'],
      dtype='object')


In [4]:
url_place = "https://www2.census.gov/geo/tiger/TIGER2020/PLACE/tl_2020_13_place.zip"

zip_place = "tl_2020_13_place.zip"
out_place = "tl_2020_13_place"

if not os.path.exists(zip_place):
    r = requests.get(url_place, stream=True)
    with open(zip_place, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            f.write(chunk)

if not os.path.exists(out_place):
    with zipfile.ZipFile(zip_place, "r") as z:
        z.extractall(out_place)

places = gpd.read_file(os.path.join(out_place, "tl_2020_13_place.shp"))

In [5]:
atlanta_city = places[places["NAME"] == "Atlanta"].copy()

# match CRS
blocks_ga = blocks_ga.to_crs(atlanta_city.crs)

# clip blocks
atlanta_blocks = gpd.clip(blocks_ga, atlanta_city)

print("Atlanta blocks:", len(atlanta_blocks))

Atlanta blocks: 6575


In [6]:
atlanta_blocks = atlanta_blocks.to_crs("EPSG:2240")
gdf_points = gdf_points.to_crs("EPSG:2240")

In [7]:
points_in_city = gpd.sjoin(
    gdf_points,
    atlanta_city.to_crs(gdf_points.crs),
    predicate="within",
    how="inner"
)

print("Points inside Atlanta:", len(points_in_city))

Points inside Atlanta: 407200


In [8]:
joined = gpd.sjoin(
    points_in_city,
    atlanta_blocks,
    how="left",
    predicate="within"
)

print("Unmatched points:", joined["GEOID20"].isna().sum())

ValueError: 'index_right' cannot be a column name in the frames being joined